# 벡터 거리와 정확 검색

이 노트북은 빈 Python 3.12 커널에서 순서대로 실행합니다.


## 실행 환경 준비

OpenAI 임베딩으로 12개 문서와 8개 질문을 같은 256차원 공간에 놓습니다. API 예상 비용은 미화 0.01달러 미만입니다.

### 거리 계산의 입력과 방향

코사인 유사도는 방향이 비슷할수록 커지고 L2 거리는 가까울수록 작아 정렬 방향이 반대입니다. 정확 검색은 모든 문서 벡터를 비교하므로 근사 인덱스의 후보 누락이 없지만, 임베딩이 질문 의미를 잘 표현했는지까지 보장하지 않습니다. 정확한 코드 문자열은 sparse 검색과 별도로 비교합니다.


In [1]:
# 이 절에서 사용하는 패키지 버전을 실습 환경과 맞춥니다.
%pip install -q "numpy==2.5.3" "python-dotenv==1.2.3" "langchain-openai==1.6.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, os
from pathlib import Path
import numpy as np
from dotenv import find_dotenv, load_dotenv
from langchain_openai import OpenAIEmbeddings

# 현재 폴더부터 상위 폴더에서 .env를 찾고 키의 존재만 확인합니다.
env_file = find_dotenv(usecwd=True)
assert env_file, "프로젝트 루트에 .env를 준비하세요."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY를 확인하세요."
course_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "labs/part3/ch12-retrieval-tuning/data").is_dir()), None)
assert course_root is not None, "Ch 12 실습 데이터 폴더를 찾지 못했습니다."
chapter_dir = course_root / "labs/part3/ch12-retrieval-tuning"
data_dir = chapter_dir / "data"
docs = json.loads((data_dir / "core-documents.json").read_text())
queries = json.loads((data_dir / "core-queries.json").read_text())
embedder = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=256)

## 기본 실행

문서와 첫 질문을 임베딩하고 벡터 길이와 노름을 확인합니다.


In [3]:
texts = [f"{doc['title']} {doc['text']}" for doc in docs]
doc_vectors = np.array(embedder.embed_documents(texts))
query_vectors = np.array([embedder.embed_query(q["question"]) for q in queries])
print("문서 벡터 모양:", doc_vectors.shape)
print("질문 벡터 모양:", query_vectors.shape)
print("첫 문서 벡터 L2 노름:", round(float(np.linalg.norm(doc_vectors[0])), 4))

문서 벡터 모양: (12, 256)
질문 벡터 모양: (8, 256)
첫 문서 벡터 L2 노름: 0.9999


## 중간 결과 관찰

전체 문서와 거리를 계산합니다. 작은 데이터의 정확 검색은 모든 행을 비교하므로 이후 근사 검색의 기준선이 됩니다.


In [4]:
def exact_rank(query_vector, metric="cosine", k=3):
    if metric == "cosine":
        scores = doc_vectors @ query_vector / (
            np.linalg.norm(doc_vectors, axis=1) * np.linalg.norm(query_vector)
        )
        order = np.argsort(-scores)[:k]
        return [(docs[i]["doc_id"], round(float(scores[i]), 4)) for i in order]
    distances = np.linalg.norm(doc_vectors - query_vector, axis=1)
    order = np.argsort(distances)[:k]
    return [(docs[i]["doc_id"], round(float(distances[i]), 4)) for i in order]

print("코사인 상위 3개:", exact_rank(query_vectors[0], "cosine"))
print("L2 상위 3개:", exact_rank(query_vectors[0], "l2"))

코사인 상위 3개: [('LIB-004', 0.5677), ('LIB-001', 0.4416), ('LIB-003', 0.3791)]
L2 상위 3개: [('LIB-004', 0.9299), ('LIB-001', 1.0566), ('LIB-003', 1.1144)]


## 값 하나를 바꿔 비교

질문과 문서는 유지하고 계산 방식만 코사인 유사도에서 L2 거리로 바꿉니다.


In [5]:
for metric in ["cosine", "l2"]:
    top_ids = [doc_id for doc_id, _ in exact_rank(query_vectors[1], metric)]
    print(metric, "순위:", top_ids, "정답 포함:", queries[1]["gold_doc_id"] in top_ids)

cosine 순위: ['LIB-003', 'LIB-004', 'LIB-001'] 정답 포함: False
l2 순위: ['LIB-003', 'LIB-004', 'LIB-001'] 정답 포함: False


## 반복 실험과 결과 해석

8개 질문에서 정답 문서가 상위 3개 안에 든 비율을 계산하고 원자료를 저장합니다.


In [6]:
rows = []
for metric in ["cosine", "l2"]:
    for query, vector in zip(queries, query_vectors):
        ranked = [doc_id for doc_id, _ in exact_rank(vector, metric)]
        rows.append({"metric": metric, "qid": query["qid"], "ranked": ranked,
                     "hit": query["gold_doc_id"] in ranked})
for metric in ["cosine", "l2"]:
    selected = [row for row in rows if row["metric"] == metric]
    print(metric, "Recall@3:", sum(row["hit"] for row in selected), "/", len(selected))
print("기록할 결과 행:", len(rows))

cosine Recall@3: 7 / 8
l2 Recall@3: 7 / 8
기록할 결과 행: 16


<h3>실행 결과 보존</h3><p>기존 정식 결과를 해시 이름으로 보존하고, 이번 결과를 실행 시각이 포함된 파일에 먼저 저장한 뒤 최신 결과 파일을 갱신합니다.</p>

In [7]:
import hashlib
from datetime import UTC, datetime
payload = json.dumps(rows, ensure_ascii=False, indent=2)
runs = chapter_dir / "runs"; history = runs / "core-history"
history.mkdir(parents=True, exist_ok=True)
pointer = runs / "core-01-exact.json"
if pointer.exists():
    old = pointer.read_bytes()
    backup = history / f"before-{hashlib.sha256(old).hexdigest()[:12]}-{pointer.name}"
    if not backup.exists(): backup.write_bytes(old)
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f")
archive = history / f"{run_id}-{pointer.name}"
archive.write_text(payload, encoding="utf-8")
pointer.write_text(payload, encoding="utf-8")
print("실행별 보존 로그:", archive.relative_to(course_root))

실행별 보존 로그: labs/part3/ch12-retrieval-tuning/runs/core-history/20260909T174506809610-core-01-exact.json
